In [42]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [43]:
import json
from pathlib import Path

import pandas as pd

from rich import print
from openai import OpenAI

In [44]:
client = OpenAI()

In [45]:
def download_job(id: str, file_path: str):
    batch = client.batches.retrieve(id)
    print(batch)

    if (batch.status == "completed"):
        file_content = client.files.content(batch.output_file_id).content
        with open(file_path, "wb") as f:
            f.write(file_content)

In [53]:
def gen_items(path: str):
    with (open(path, "r") as batch_file):
        for line in batch_file:
            parsed = json.loads(line)
            json_content = parsed["response"]["body"]["choices"][0]["message"]["content"]

            try:
                structured_output = json.loads(json_content)
                total_tokens = parsed["response"]["body"]["usage"]["total_tokens"]
                custom_id = parsed["custom_id"]

                yield {**{k.replace("_english", ""):v for k, v in structured_output.items()}, "custom_id": custom_id, "total_tokens": total_tokens}
            except:
                print("FAILED CUSTOM ID: " + parsed["custom_id"])

## Triplet

In [47]:
triplet_job_path = Path("../data/llm-gen/translated/triplet_batch_result.jsonl")
download_job("batch_6817fe84fd34819084b7c63b5bc49b69", triplet_job_path)

Batch(
    id='batch_6817fe84fd34819084b7c63b5bc49b69',
    completion_window='24h',
    created_at=1746402948,
    endpoint='/v1/chat/completions',
    input_file_id='file-3EWAaLmfUFgXTtu1psVV58',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1746405033,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746489348,
    failed_at=None,
    finalizing_at=1746404501,
    in_progress_at=1746402952,
    metadata=None,
    output_file_id='file-QjDdzGW9GNytgZaYNBEQj6',
    request_counts=BatchRequestCounts(completed=7492, failed=0, total=7492)
)

In [48]:
df_triplet = pd.read_json("../data/cleaned/triplet.jsonl", lines=True).reset_index()
df_triplet.head()

,index,query,positive,negative
0,0,Kumaha cara ngahontal kahayang dina kahirupan?,"Kahiji, urang kedah sabar sareng henteu janten...",Abdi ngadangu seueur warta ngeunaan jalma anu ...
1,1,Naon anu kedah dilakukeun lamun gering?,"Lamun gering, ulah rungsing sabab pikiran posi...",Masyarakat ayeuna seueur nganggur di kota nu g...
2,2,Kumaha cara nyieun amal?,"Ngawitan amal ti hal-hal leutik, sapertos ngab...",Jalma sering nyarita ngeunaan kaékonomian anu ...
3,3,Kumaha sangkan ngeterkeun diri ka Gusti?,"Mertahankeun ati, pariksa tindakan sorangan, s...","Saurang guru ngajarkeun pentingna ilmu, tapi k..."
4,4,Naon hartina jadi jalma leutik?,Jadi jalma leutik hartina ulah sombong sareng ...,"Dina pagelaran, anu katinggali gaduh prestasi ..."


In [49]:
df_triplet_map = pd.read_json("../data/llm-gen/translated/triplet_map.jsonl", lines=True)
df_triplet_map.head()

,custom_id,doc_index
0,d822f175-d420-43c6-8d94-3a1338ad90f3,0
1,8388186f-2934-4399-8d18-e57be48bedc8,1
2,b154854f-c071-4532-9974-22c3330c1dd3,2
3,81941833-084f-4f0f-9d02-7648e468a1c3,3
4,24e236ca-a6a2-48a3-b86d-0f91f90baf47,4


In [54]:
df_triplet_en = pd.DataFrame(gen_items(triplet_job_path))
df_triplet_en.head()

FAILED CUSTOM ID: d0d6caa8-c6ea-4794-916d-fa03c27a1b9c

,query,positive,negative,custom_id,total_tokens
0,How to achieve desires in life?,"First, we must be patient and not become compl...",I have heard a lot of news about successful pe...,d822f175-d420-43c6-8d94-3a1338ad90f3,291
1,What should be done if you are sick?,"If you are sick, don't worry because positive ...",Many people are currently unemployed in big ci...,8388186f-2934-4399-8d18-e57be48bedc8,283
2,How to make charity?,"Starting charity from small things, like shari...","People often talk about a better economy, but ...",b154854f-c071-4532-9974-22c3330c1dd3,279
3,How to connect oneself to God?,"Maintain your heart, examine your own actions,...","A teacher teaches the importance of knowledge,...",81941833-084f-4f0f-9d02-7648e468a1c3,286
4,What does it mean to be a humble person?,Being a humble person means not being arrogant...,"In performances, those who appear to have high...",24e236ca-a6a2-48a3-b86d-0f91f90baf47,289


In [55]:
print(f"Total tokens: {df_triplet_en['total_tokens'].sum()}")

Total tokens: 2054457

In [58]:
df_triplet_merged = df_triplet[["index"]].merge(df_triplet_map, left_on="index", right_on="doc_index")
df_triplet_merged = df_triplet_merged.merge(df_triplet_en, on="custom_id")
df_triplet_merged = df_triplet_merged.drop(columns=["index", "doc_index", "custom_id", "total_tokens"])
df_triplet_merged.head()

,query,positive,negative
0,How to achieve desires in life?,"First, we must be patient and not become compl...",I have heard a lot of news about successful pe...
1,What should be done if you are sick?,"If you are sick, don't worry because positive ...",Many people are currently unemployed in big ci...
2,How to make charity?,"Starting charity from small things, like shari...","People often talk about a better economy, but ..."
3,How to connect oneself to God?,"Maintain your heart, examine your own actions,...","A teacher teaches the importance of knowledge,..."
4,What does it mean to be a humble person?,Being a humble person means not being arrogant...,"In performances, those who appear to have high..."


In [59]:
df_triplet_merged.to_json("../data/cleaned/triplet_en.jsonl", orient="records", lines=True)

## BEIR

### Queries

In [60]:
beir_queries_job_path = Path("../data/llm-gen/translated/beir_queries_batch_result.jsonl")
download_job("batch_6817fe8d33a481908971f8c1db7abc59", beir_queries_job_path)

Batch(
    id='batch_6817fe8d33a481908971f8c1db7abc59',
    completion_window='24h',
    created_at=1746402957,
    endpoint='/v1/chat/completions',
    input_file_id='file-5uM5xHmGg3S1hHQdsoWDAw',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1746408052,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746489357,
    failed_at=None,
    finalizing_at=1746406855,
    in_progress_at=1746402961,
    metadata=None,
    output_file_id='file-QziDPNoLwvfwxZ6AQfAwmD',
    request_counts=BatchRequestCounts(completed=7491, failed=0, total=7491)
)

In [61]:
df_queries = pd.read_json("../data/cleaned/queries.jsonl", lines=True).reset_index()
df_queries.head()

,index,_id,text
0,0,f4ee6408-7047-498e-a319-9189cf81d378,apa maksud dari bismillah yuga lampah
1,1,c827cfdb-2c57-486f-abc7-80355b884699,kenapa penting sabar dalam hidup
2,2,0dce18d7-b0b8-4202-a1d5-683489ac423d,apa yang dimaksud dengan halangan dalam mencap...
3,3,5c961ec5-158a-4073-b33e-3a0be48de7a0,bagaimana cara menjadi orang yang baik menurut...
4,4,3a019977-ed9b-4d71-8dbd-a65c1c95cc65,apa arti dari pariksa ati


In [62]:
df_queries_map = pd.read_json("../data/llm-gen/translated/beir_query_map.jsonl", lines=True)
df_queries_map.head()

,custom_id,doc_index
0,3dab088a-03be-4630-a59d-d3f464bfacd4,0
1,e28bb9c1-7542-4013-b05f-9765b67c7ec8,1
2,40eb9340-fa6f-4a98-89f5-028b11b83def,2
3,e9ccad2f-5564-40f2-8d87-4e86d1fe4350,3
4,6497bd9f-8782-451c-ab3a-1742faa567d7,4


In [63]:
df_queries_en = pd.DataFrame(gen_items(beir_queries_job_path))
df_queries_en.head()

,query,custom_id,total_tokens
0,"What does ""bismillah yuga lampah"" mean?",3dab088a-03be-4630-a59d-d3f464bfacd4,127
1,Why is patience important in life?,e28bb9c1-7542-4013-b05f-9765b67c7ec8,117
2,What is meant by obstacles in reaching dreams?,40eb9340-fa6f-4a98-89f5-028b11b83def,124
3,how to become a good person according to the t...,e9ccad2f-5564-40f2-8d87-4e86d1fe4350,123
4,What is the meaning of 'pariksa ati'?,6497bd9f-8782-451c-ab3a-1742faa567d7,120


In [64]:
print(f"Total tokens: {df_queries_en['total_tokens'].sum()}")

Total tokens: 921062

In [65]:
df_queries_merged = df_queries[["index", "_id"]].merge(df_queries_map, left_on="index", right_on="doc_index")
df_queries_merged = df_queries_merged.merge(df_queries_en, on="custom_id")
df_queries_merged = df_queries_merged.drop(columns=["index", "doc_index", "custom_id", "total_tokens"])
df_queries_merged.head()

,_id,query
0,f4ee6408-7047-498e-a319-9189cf81d378,"What does ""bismillah yuga lampah"" mean?"
1,c827cfdb-2c57-486f-abc7-80355b884699,Why is patience important in life?
2,0dce18d7-b0b8-4202-a1d5-683489ac423d,What is meant by obstacles in reaching dreams?
3,5c961ec5-158a-4073-b33e-3a0be48de7a0,how to become a good person according to the t...
4,3a019977-ed9b-4d71-8dbd-a65c1c95cc65,What is the meaning of 'pariksa ati'?


In [66]:
df_queries_merged.to_json("../data/cleaned/queries_en.jsonl", orient="records", lines=True)

### Corpus

In [67]:
beir_corpus_job_path = Path("../data/llm-gen/translated/beir_corpus_batch_result.jsonl")
download_job("batch_68180c78133881908cb8388e303a966e", beir_corpus_job_path)

Batch(
    id='batch_68180c78133881908cb8388e303a966e',
    completion_window='24h',
    created_at=1746406520,
    endpoint='/v1/chat/completions',
    input_file_id='file-G1fyzVU2mi2NXceLGPyQbQ',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1746409309,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746492920,
    failed_at=None,
    finalizing_at=1746409134,
    in_progress_at=1746406522,
    metadata=None,
    output_file_id='file-RPKHKA6RY4vWBuE2bzx5kE',
    request_counts=BatchRequestCounts(completed=1499, failed=0, total=1499)
)

In [68]:
df_corpus = pd.read_json("../data/cleaned/corpus.jsonl", lines=True).reset_index()
df_corpus.head()

,index,_id,title,text
0,0,0f438470-de7f-47cb-8ba2-16e8b1ff5750,JEMBAR SABAR,bismillah yuga lampah balukar janglar meunang ...
1,1,c51ddd60-adc7-4b95-b09d-3c4865ba2aaf,KANYERI,eweuh deui seri nu lewih nyeri tibatan di héna...
2,2,cb4eb9c6-bf7f-4edc-b997-73a02780e60d,KARUNGING LAIN WAYAH,wanci gayuh ka peuting poék tungkeb haté lain ...
3,3,af9fc2c2-0230-4784-8104-cf9271ccfccc,KUDU DAÉK GAWÉ,ieu awak asa lalungsé di rasa asa carapé meure...
4,4,2ae6c860-37c6-47f0-aefc-8b74f96b623b,PUISI PANGGGEUING DIRI,naha anjeun téh poho yén mot téh dodoho datang...


In [69]:
df_corpus_map = pd.read_json("../data/llm-gen/translated/beir_corpus_map.jsonl", lines=True)
df_corpus_map.head()

,custom_id,doc_index
0,41e2f3cd-4c56-40d3-876c-1cf0b83c35fb,0
1,134c29ff-604b-4c87-8b86-ac3ef807af04,1
2,1b1b0ab6-e6ac-4b66-94ab-98e2c89cabfa,2
3,10f8ced3-41ae-4bda-a435-067b84ab2ce0,3
4,5c415019-f192-415f-91e0-e97c756232b0,4


In [70]:
df_corpus_en = pd.DataFrame(gen_items(beir_corpus_job_path))
df_corpus_en.head()

FAILED CUSTOM ID: a71550fa-4d9c-4c16-8d8d-e9b578cacf24

,title,body,custom_id,total_tokens
0,WIDE PATIENCE,"In the name of Allah, may we act with the resu...",41e2f3cd-4c56-40d3-876c-1cf0b83c35fb,373
1,HURT,There is no other pain that is greater than th...,134c29ff-604b-4c87-8b86-ac3ef807af04,426
2,THE OTHER SIDE OF THE NIGHT,"as the evening approaches, heart tightly wrapp...",1b1b0ab6-e6ac-4b66-94ab-98e2c89cabfa,315
3,MUST BE WILLING TO WORK,"This body feels a sense of heaviness, as if bu...",10f8ced3-41ae-4bda-a435-067b84ab2ce0,304
4,POEM OF SELF REMEMBERING,"do you forget that death comes unannounced, no...",5c415019-f192-415f-91e0-e97c756232b0,388


In [71]:
print(f"Total tokens: {df_corpus_en['total_tokens'].sum()}")

Total tokens: 1104142

In [72]:
df_corpus_merged = df_corpus[["index", "_id"]].merge(df_corpus_map, left_on="index", right_on="doc_index")
df_corpus_merged = df_corpus_merged.merge(df_corpus_en, on="custom_id")
df_corpus_merged = df_corpus_merged.drop(columns=["index", "doc_index", "custom_id", "total_tokens"])
df_corpus_merged.head()

,_id,title,body
0,0f438470-de7f-47cb-8ba2-16e8b1ff5750,WIDE PATIENCE,"In the name of Allah, may we act with the resu..."
1,c51ddd60-adc7-4b95-b09d-3c4865ba2aaf,HURT,There is no other pain that is greater than th...
2,cb4eb9c6-bf7f-4edc-b997-73a02780e60d,THE OTHER SIDE OF THE NIGHT,"as the evening approaches, heart tightly wrapp..."
3,af9fc2c2-0230-4784-8104-cf9271ccfccc,MUST BE WILLING TO WORK,"This body feels a sense of heaviness, as if bu..."
4,2ae6c860-37c6-47f0-aefc-8b74f96b623b,POEM OF SELF REMEMBERING,"do you forget that death comes unannounced, no..."


In [73]:
df_corpus_merged.to_json("../data/cleaned/corpus_en.jsonl", orient="records", lines=True)